# functools : Higher-Order Functions & Callable Utilities

## 1. `functools.cache`

`@cache` provides a lightweight, unbounded memoization cache. It is equivalent to an unbounded `lru_cache`.

Use it when:
- the function is deterministic for the same arguments;
- arguments are hashable;
- keeping all cached results is acceptable.


In [ ]:
from functools import cache

calls = 0

@cache
def fibonacci(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(20))
print("Underlying calls:", calls)
print(fibonacci(20))  # cached
print("Underlying calls:", calls)
print(fibonacci.cache_info())

6765
Underlying calls: 21
6765
Underlying calls: 21
CacheInfo(hits=19, misses=21, maxsize=None, currsize=21)


## 2. `functools.lru_cache`

`lru_cache` stores recent calls and can limit cache size.

Important utilities on the wrapped function:
- `cache_info()`
- `cache_clear()`
- `cache_parameters()`

Use `maxsize=None` when we want an unbounded cache; otherwise the least-recently-used entries can be evicted.


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=2)
def square(n):
    print("computing", n)
    return n * n

print(square(2))
print(square(3))
print(square(2)) # cached
print(square.cache_info())
print(square.cache_parameters())

square.cache_clear() # clear the cache
print(square.cache_info()) 

computing 2
4
computing 3
9
4
CacheInfo(hits=1, misses=2, maxsize=2, currsize=2)
{'maxsize': 2, 'typed': False}
CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)


## 3. `cached_property`

`cached_property` turns a method into a property whose value is computed once and then cached on the instance.

It is useful for expensive, immutable per-instance computations.


In [ ]:
from functools import cached_property

class Dataset:
    def __init__(self, values):
        self.values = values

    @cached_property
    def total(self):
        print("Calculating total...")
        return sum(self.values)

data = Dataset([10, 20, 30])

print(data.total)
print(data.total)  # uses cached value

Calculating total...
60
60


## 4. `partial`

`partial(func, *args, **keywords)` creates a new callable with some arguments pre-filled.

This is called **partial application**.

It is useful for:
- adapting APIs;
- creating specialized versions of a general function;
- passing configured callables to higher-order functions.


In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)

print(square(5))
print(cube(4))

print(square.func) # prints the original function that was partially applied
print(square.keywords) # prints the keyword arguments that were partially applied

25
64
<function power at 0x000001C64556FB60>
{'exponent': 2}


## 5. `partial` with positional arguments

Extra arguments supplied later are appended to the pre-filled positional arguments.


In [ ]:
from functools import partial

def multiply(a, b, c):
    return a * b * c

double_product = partial(multiply, 2, 10) # create a new function that multiplies by 2 and 10

print(double_product(5)) # This will compute 2 * 10 * 5 = 100

100


## 6. `partialmethod`

`partialmethod` is designed for defining methods with pre-filled arguments.

It is especially useful when a class exposes several named operations that are variations of one general method.


In [7]:
from functools import partialmethod

class Cell:
    def __init__(self):
        self.state = False

    def set_state(self, value):
        self.state = bool(value)

    set_alive = partialmethod(set_state, True)
    set_dead = partialmethod(set_state, False)

cell = Cell()
cell.set_alive()
print(cell.state)
cell.set_dead()
print(cell.state)

True
False


## 7. `reduce`

`reduce(function, iterable[, initial])` repeatedly combines items into one accumulated result.

Use it when a reduction is genuinely clearer than a loop or a built-in such as `sum`, `min`, `max`, or `all`.

The accumulator evolves like:

`(((first ⊗ second) ⊗ third) ⊗ fourth)`



In [8]:
from functools import reduce

numbers = [1, 2, 3, 4]

product = reduce(lambda a, b: a * b, numbers)
product_with_initial = reduce(lambda a, b: a * b, numbers, 10)

print(product)
print(product_with_initial)

24
240


## 8. `cmp_to_key`

Older APIs sometimes use comparison functions taking two arguments and returning negative/zero/positive.

Modern sorting APIs generally prefer a one-argument `key` function. `cmp_to_key` adapts a comparison function for APIs such as `sorted()` and `list.sort()`.


In [9]:
from functools import cmp_to_key

def compare_length(a, b):
    return (len(a) > len(b)) - (len(a) < len(b))

words = ["pear", "fig", "banana", "kiwi"]
print(sorted(words, key=cmp_to_key(compare_length)))

['fig', 'pear', 'kiwi', 'banana']


## 9. `singledispatch`

`singledispatch` creates a generic function whose implementation is selected based on the type of the **first argument**.

The base implementation handles unmatched types; registered implementations specialize behavior.


In [10]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"object: {value!r}"

@describe.register
def _(value: int):
    return f"integer: {value}"

@describe.register
def _(value: list):
    return f"list with {len(value)} items"

print(describe(10))
print(describe([1, 2, 3]))
print(describe("hello"))

print(describe.registry.keys())

integer: 10
list with 3 items
object: 'hello'
dict_keys([<class 'object'>, <class 'int'>, <class 'list'>])


### Explicit registration

we can also register a type explicitly:

```python
@describe.register(float)
def describe_float(value):
    ...
```

This can be useful when annotations are not the right fit.


In [11]:
@describe.register(float)
def describe_float(value):
    return f"float: {value:.2f}"

print(describe(3.14159))

float: 3.14


## 10. `singledispatchmethod`

This is the method-oriented version of `singledispatch`. Dispatch is based on the first non-`self`/`cls` argument.


In [12]:
from functools import singledispatchmethod

class Formatter:
    @singledispatchmethod
    def format(self, value):
        return f"generic:{value}"

    @format.register
    def _(self, value: int):
        return f"int:{value + 1}"

    @format.register
    def _(self, value: list):
        return f"list:{len(value)}"

f = Formatter()
print(f.format("x"))
print(f.format(10))
print(f.format([1, 2, 3]))

generic:x
int:11
list:3


## 11. `total_ordering`

`total_ordering` is a class decorator that can supply the remaining ordering methods when a class defines `__eq__` and one ordering operation such as `__lt__`.

Trade-off: it reduces boilerplate but can be slower than implementing all comparison methods directly.


In [14]:
from functools import total_ordering

@total_ordering
class Score:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if not isinstance(other, Score):
            return NotImplemented
        return self.value == other.value

    def __lt__(self, other):
        if not isinstance(other, Score):
            return NotImplemented
        return self.value < other.value

a = Score(10)
b = Score(20)

print(a < b)
print(a <= b)
print(a > b)
print(a >= b)
print(a == b)
print(a != b)

True
True
False
False
False
True


## 12. `update_wrapper` and `wraps`

`update_wrapper(wrapper, wrapped, ...)` copies selected metadata from one callable to another.

`wraps(wrapped, ...)` is a decorator factory that makes it convenient to apply `update_wrapper`.

Use these when writing decorators so that the wrapper retains useful metadata.


In [15]:
from functools import update_wrapper, wraps

def original(x):
    """Important documentation."""
    return x * 2

def wrapper(x):
    return original(x)

update_wrapper(wrapper, original)

print(wrapper.__name__)
print(wrapper.__doc__)

original
Important documentation.


In [16]:
def logging_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@logging_decorator
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(2, 3))
print(add.__name__)
print(add.__doc__)

calling add
5
add
Add two numbers.


## 13. Choosing the right `functools` tool

| Tool | Main idea | Typical use |
|---|---|---|
| `cache` | unbounded memoization | repeated deterministic calls |
| `lru_cache` | bounded/recent-call memoization | expensive repeated calls |
| `cached_property` | cache an instance property | expensive per-object calculation |
| `partial` | pre-fill arguments | adapt/configure callables |
| `partialmethod` | pre-fill method arguments | class API variants |
| `reduce` | fold many values into one | custom reductions |
| `cmp_to_key` | adapt 2-arg comparison | legacy comparison functions |
| `singledispatch` | dispatch by first argument type | generic functions |
| `singledispatchmethod` | dispatch method by argument type | generic methods |
| `total_ordering` | generate comparison methods | reduce comparison boilerplate |
| `update_wrapper` | copy callable metadata | custom decorators |
| `wraps` | convenient `update_wrapper` | decorator wrappers |
| `Placeholder` | reserve positional slots | advanced `partial` |
